already edited via NotebookEdit

already edited via NotebookEdit

# L29 · 随机的形状 — 常见概率分布

高斯分布在自然界中无处不在，这是中心极限定理的直接结果：大量独立随机变量之和趋向正态分布。神经网络权重初始化（Xavier/He 初始化）用正态分布确保每层激活方差稳定；VAE 的隐变量先验 N(0,1) 让隐空间连续可插值；扩散模型的加噪过程逐步叠加高斯噪声。理解正态分布的形状和参数，是读懂这些算法的前提。

## 实验入口：采样如何收敛到分布形状

生成 N=10 和 N=10000 个样本，观察直方图的轮廓：小样本时形状粗糙，大样本时逼近理论 PDF 曲线。

In [ ]:
import numpy as np, matplotlib.pyplot as plt
rng = np.random.default_rng(0)
u = rng.uniform(-3, 3, 5000)    # 均匀: 区间内处处等可能
g = rng.normal(0, 1, 5000)      # 正态: 中间多、两边少
fig, ax = plt.subplots(1, 2, figsize=(9,3))
ax[0].hist(u, bins=40); ax[0].set_title('uniform')
ax[1].hist(g, bins=40); ax[1].set_title('normal'); plt.show()

## 动手观察：随机代码要多试几次

多次运行下面的格，观察小样本（N=10）的均值每次波动多大，大样本（N=10000）的均值有多稳定。

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
for n in [10, 100, 10_000]:
    rolls = rng.integers(1, 7, size=n)
    estimate = np.mean(rolls == 6)
    print(f'n={n:5d}  估计 P(掷到6) = {estimate:.3f}')


## 代码实验：多次重复同一个随机实验

这个实验展示样本量越大、重复实验之间均值的方差越小——这是大数定律的直接体现。

In [ ]:
import numpy as np

for n in [20, 200, 2000]:
    estimates = []
    for seed in range(5):
        rng = np.random.default_rng(seed)
        rolls = rng.integers(1, 7, size=n)
        estimates.append(np.mean(rolls == 6))
    print(f'n={n:4d} ->', np.round(estimates, 3), '平均=', round(float(np.mean(estimates)), 3))


## 2. ✏️ 实现正态分布概率密度 `gaussian_pdf(x, mu, sigma)`

`pdf = exp(−(x−μ)² / (2σ²)) / (σ·√(2π))`

**推理路线**：
1. 计算指数项：`exponent = -((x - mu)**2) / (2 * sigma**2)`。分子是偏离均值的平方（总为非负），`sigma**2` 控制衰减速度——sigma 越大衰减越慢。
2. 计算归一化系数：`norm = sigma * np.sqrt(2 * np.pi)`。这个分母保证整条曲线积分恰好为 1，无论 sigma 取何值。
3. 返回 `np.exp(exponent) / norm`。NumPy 自动广播：x 是数组时逐元素计算，是标量时返回标量。

**参考输入输出**：`mu=0, sigma=1, x=0` → pdf=1/√(2π)≈0.399；`x=1` → pdf≈0.242（下降到峰值的 60.7%）

<details><summary>点击查看参考实现</summary>

```python
def gaussian_pdf(x, mu, sigma):
    exponent = -((x - mu) ** 2) / (2 * sigma ** 2)
    normalization = sigma * np.sqrt(2 * np.pi)
    return np.exp(exponent) / normalization
```

</details>

**提示**：用 `np.exp`、`np.sqrt`、`np.pi`。

写 `gaussian_pdf` 前明确三件事：
- 输入：`x`（评估点，数组），`mu`（均值），`sigma`（标准差，须 > 0）
- 关键步骤：计算指数项 `exp(-(x-mu)**2 / (2*sigma**2))`，再除以归一化系数 `sigma * sqrt(2π)`
- 返回：与 `x` 形状相同的 PDF 值数组，所有值均非负

In [ ]:
def gaussian_pdf(x, mu=0.0, sigma=1.0):
    # ✏️ TODO: 返回正态分布在 x 处的概率密度
    return None  # ← 改这里


In [ ]:
import numpy as np
xs = np.linspace(-4, 4, 9)
vals = gaussian_pdf(xs, 0.0, 1.0)
peak = gaussian_pdf(0.0)
assert abs(peak - 1/np.sqrt(2*np.pi)) < 1e-9, '峰值应在 μ 处'
assert abs(gaussian_pdf(1.0) - gaussian_pdf(-1.0)) < 1e-12, '应关于 μ 对称'
print('✅ 通过：你实现了高斯密度，峰值=', round(peak,4))

**🔗 Aurora 连接**：`gaussian_pdf` 是 Aurora 生成流水线的密度评估原语。He 权重初始化（`np.random.normal(0, std, shape)`）、VAE 隐变量采样（`z ~ N(0, I)`）、扩散模型加噪（逐步叠加 `N(0, β_t·I)` 噪声）——三处都直接对应今天手写的公式。


In [ ]:
for n in [20, 100, 1000, 10000]:
    rng = np.random.default_rng(7)
    rolls = rng.integers(1, 7, size=n)
    estimate = np.mean(rolls == 6)
    error = abs(estimate - 1/6)
    print(f'n={n:5d} | 估计={estimate:.4f} | 离理论值差={error:.4f}')


## 参数实验：只改一个旋钮

固定 `mu=0`，把 `sigma` 从 0.5 改到 2，观察峰值从 0.8 降到 0.2（更扁平），但曲线下面积始终为 1。这说明 sigma 控制"集中程度"，不改变总概率。再把 `mu` 从 0 改到 2，曲线整体平移但形状（峰值、宽度）不变——mu 只决定分布的中心位置。

In [ ]:
for n in [20, 100, 1000, 10000]:
    rng = np.random.default_rng(7)
    rolls = rng.integers(1, 7, size=n)
    estimate = np.mean(rolls == 6)
    error = abs(estimate - 1/6)
    print(f'n={n:5d} | 估计={estimate:.4f} | 离理论值差={error:.4f}')


## 本课收束

现在可以用 `gaussian_pdf(x, mu, sigma)` 在任意点求正态分布的密度值，并通过 `np.random.uniform` / `np.random.normal` 生成两种分布的样本。`gaussian_pdf` 对应 Aurora 中加噪步骤的密度评估和 VAE 的 KL 散度计算。下一节用 softmax 把未归一化得分变成概率分布，今天手写的归一化系数思路在那里会再出现。

In [ ]:
# 小检查：同一个实验，样本量越大越稳定
import numpy as np

for n in [30, 300, 3000]:
    rng = np.random.default_rng(42)
    samples = rng.integers(1, 7, size=n)
    estimate = np.mean(samples == 6)
    print(f'n={n:4d} -> P(6)≈{estimate:.3f}')
